# CNN-GRU-SPN-Gamma – GEO vs TIME

In [1]:
!nvidia-smi


Wed Mar 11 21:14:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip -q install -U kagglehub==1.0.0 scipy scikit-learn pandas matplotlib tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 142.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 19.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pand

In [3]:
%cd /content
!rm -rf cnn-gru-spn-gamma-temperature-forecast
!git clone https://github.com/MariaAlexandraBadea/cnn-gru-spn-gamma-temperature-forecast.git
%cd cnn-gru-spn-gamma-temperature-forecast
!python -m py_compile cnn_gru_spn_gamma_temperature_forecast.py


/content
Cloning into 'cnn-gru-spn-gamma-temperature-forecast'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 46 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 149.64 KiB | 1.20 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/cnn-gru-spn-gamma-temperature-forecast


In [4]:
from pathlib import Path
import re

p = Path("cnn_gru_spn_gamma_temperature_forecast.py")
s = p.read_text(encoding="utf-8", errors="ignore")

# --- A) Ensure sys imported + CLI args are actually read from sys.argv ---
if re.search(r"^\s*import\s+sys\b", s, flags=re.M) is None:
    m = re.search(r"^(?:from __future__ import .*?\n)+", s, flags=re.M)
    if m:
        s = s[:m.end()] + "import sys\n" + s[m.end():]
    else:
        s = "import sys\n" + s

s = s.replace(
    "return p.parse_args(args=([] if argv is None else argv))",
    "return p.parse_args(args=(sys.argv[1:] if argv is None else argv))"
)

# --- B) Add CLI args (env/epochs/patience/batch) if missing ---
if 'p.add_argument("--env"' not in s:
    s = s.replace(
        'p.add_argument("--split_seed_base", type=int, default=1000)',
        'p.add_argument("--split_seed_base", type=int, default=1000)\n'
        '    p.add_argument("--env", type=str, default="geo", choices=["geo","time"])\n'
        '    p.add_argument("--epochs", type=int, default=None)\n'
        '    p.add_argument("--patience", type=int, default=None)\n'
        '    p.add_argument("--batch", type=int, default=None)'
    )

# --- C) Override globals from CLI + print env ---
needle = "def main(argv=None) -> None:\n    args = parse_args(argv)\n"
if needle in s and "FORCE runtime hyperparams from CLI" not in s:
    inject = needle + (
        "    # ---- FORCE runtime hyperparams from CLI ----\n"
        "    global EPOCHS, PATIENCE, BATCH\n"
        "    if getattr(args, 'epochs', None) is not None:\n"
        "        EPOCHS = int(getattr(args, 'epochs'))\n"
        "    if getattr(args, 'patience', None) is not None:\n"
        "        PATIENCE = int(getattr(args, 'patience'))\n"
        "    if getattr(args, 'batch', None) is not None:\n"
        "        BATCH = int(getattr(args, 'batch'))\n"
        "    print(f\"[DIAG] ENV mode: {getattr(args,'env','geo')}\")\n"
    )
    s = s.replace(needle, inject, 1)

# --- D) Include env in tag ---
s = s.replace(
    'tag = f"set{eval_set_id}_seedS{seed_split}_seedM{seed_model}_H{H}"',
    'tag = f"{getattr(args,\'env\',\'geo\')}_set{eval_set_id}_seedS{seed_split}_seedM{seed_model}_H{H}"'
)

# --- E) TIME-only support inside make_windows_global_joint ---
if "def make_windows_global_joint" in s:
    m_def = re.search(r"def\s+make_windows_global_joint\s*\(", s)
    if m_def:
        start = m_def.start()
        end = s.find("):", start)
        if end == -1:
            end = s.find(") ->", start)
        if end != -1:
            sig = s[start:end]
            if "env=" not in sig and "eval_set_norm" not in sig:
                idx = sig.find("max_pchip_gap")
                if idx != -1:
                    line_end = sig.find("\n", idx)
                    if line_end != -1:
                        sig_new = sig[:line_end+1] + '    env="geo",\n    eval_set_norm=None,\n' + sig[line_end+1:]
                        s = s[:start] + sig_new + s[end:]

if "TIME-only: all cities contribute to TRAIN/VAL by years" not in s:
    block_old = (
        "        if city_norm in test_cities_norm:\n"
        "            which_city = \"test\"\n"
        "        elif city_norm in val_cities_norm:\n"
        "            which_city = \"val\"\n"
        "        elif city_norm in train_cities_norm:\n"
        "            which_city = \"train\"\n"
        "        else:\n"
        "            continue\n"
    )
    block_new = (
        "        env_local = (env or \"geo\").lower()\n"
        "        is_eval = (eval_set_norm is not None and city_norm in eval_set_norm)\n"
        "\n"
        "        if env_local == \"time\":\n"
        "            # TIME-only: all cities contribute to TRAIN/VAL by years; TEST only for eval cities\n"
        "            which_city = \"time\"\n"
        "        else:\n"
        "            if city_norm in test_cities_norm:\n"
        "                which_city = \"test\"\n"
        "            elif city_norm in val_cities_norm:\n"
        "                which_city = \"val\"\n"
        "            elif city_norm in train_cities_norm:\n"
        "                which_city = \"train\"\n"
        "            else:\n"
        "                continue\n"
    )
    if block_old in s:
        s = s.replace(block_old, block_new, 1)

if "enforce split (TIME-only uses years" not in s:
    split_old = (
        "            # enforce split by city + time (using horizon end)\n"
        "            if which_city == \"train\":\n"
        "                if y_year_last > train_end:\n"
        "                    continue\n"
        "                split_key = \"train\"\n"
        "            elif which_city == \"val\":\n"
        "                if not (y_year_first > train_end and y_year_last <= val_end):\n"
        "                    continue\n"
        "                split_key = \"val\"\n"
        "            else:  # test\n"
        "                if y_year_first <= val_end:\n"
        "                    continue\n"
        "                split_key = \"test\"\n"
    )
    split_new = (
        "            # enforce split (TIME-only uses years; GEO uses city buckets + years)\n"
        "            if env_local == \"time\":\n"
        "                if y_year_last <= train_end:\n"
        "                    split_key = \"train\"\n"
        "                elif (y_year_first > train_end and y_year_last <= val_end):\n"
        "                    split_key = \"val\"\n"
        "                elif is_eval and (y_year_first > val_end):\n"
        "                    split_key = \"test\"\n"
        "                else:\n"
        "                    continue\n"
        "            else:\n"
        "                if which_city == \"train\":\n"
        "                    if y_year_last > train_end:\n"
        "                        continue\n"
        "                    split_key = \"train\"\n"
        "                elif which_city == \"val\":\n"
        "                    if not (y_year_first > train_end and y_year_last <= val_end):\n"
        "                        continue\n"
        "                    split_key = \"val\"\n"
        "                else:  # test\n"
        "                    if y_year_first <= val_end:\n"
        "                        continue\n"
        "                    split_key = \"test\"\n"
    )
    if split_old in s:
        s = s.replace(split_old, split_new, 1)

call_old = "max_pchip_gap=MAX_PCHIP_GAP,\n    )"
call_new = "max_pchip_gap=MAX_PCHIP_GAP,\n        env=getattr(args,'env','geo'),\n        eval_set_norm=eval_set_norm,\n    )"
if call_old in s and call_new not in s:
    s = s.replace(call_old, call_new, 1)

p.write_text(s, encoding="utf-8")
print("OK: patch v5 applied (fix TypeError env + TIME-only + separate tag).")
!python -m py_compile cnn_gru_spn_gamma_temperature_forecast.py



OK: patch v5 applied (fix TypeError env + TIME-only + separate tag).


In [5]:
# GEO (5 eval sets x 3 seeds)
!python cnn_gru_spn_gamma_temperature_forecast.py --env geo --horizon 6 --window 12 --seeds 42,43,44 --n_eval_sets 5 --epochs 200 --patience 8 --batch 512

[DIAG] ENV mode: geo
=== Reproducibility (ENV) ==
Device: cuda
Torch: 2.10.0+cu128 | NumPy: 2.0.2 | Pandas: 3.0.1
KAGGLE_DATASET_REF: sudalairajkumar/daily-temperature-of-major-cities
Seeds to run: [42, 43, 44]
/content/cnn-gru-spn-gamma-temperature-forecast/cnn_gru_spn_gamma_temperature_forecast.py:214: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(
100% 12.9M/12.9M [00:00<00:00, 21.7MB/s]
Extracting zip of city_temperature.csv...
/content/cnn-gru-spn-gamma-temperature-forecast/cnn_gru_spn_gamma_temperature_forecast.py:269: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  merged = full.merge(
[DIAG] Candidate cities with >= 60 TEST windows: 311
  - Africa: 27
  - Asia: 35
  - Austra

In [6]:
# Save GEO outputs separately (so they are not overwritten by the TIME run)
!test -f runs_all.csv && mv runs_all.csv runs_all_geo.csv
!test -f runs_by_evalset.csv && mv runs_by_evalset.csv runs_by_evalset_geo.csv
!test -f boxplot_rmse_evalsets.png && mv boxplot_rmse_evalsets.png boxplot_rmse_evalsets_geo.png
!test -f boxplot_energy_evalsets.png && mv boxplot_energy_evalsets.png boxplot_energy_evalsets_geo.png

# archive the artifacts folder
!rm -rf artifacts_geo
!test -d artifacts && mv artifacts artifacts_geo


In [7]:
# TIME-only (5 eval sets x 3 seeds)
!python cnn_gru_spn_gamma_temperature_forecast.py --env time --horizon 6 --window 12 --seeds 42,43,44 --n_eval_sets 5 --epochs 200 --patience 8 --batch 512

[DIAG] ENV mode: time
=== Reproducibility (ENV) ==
Device: cuda
Torch: 2.10.0+cu128 | NumPy: 2.0.2 | Pandas: 3.0.1
KAGGLE_DATASET_REF: sudalairajkumar/daily-temperature-of-major-cities
Seeds to run: [42, 43, 44]
/content/cnn-gru-spn-gamma-temperature-forecast/cnn_gru_spn_gamma_temperature_forecast.py:214: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(
Using Colab cache for faster access to the 'daily-temperature-of-major-cities' dataset.
/content/cnn-gru-spn-gamma-temperature-forecast/cnn_gru_spn_gamma_temperature_forecast.py:269: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  merged = full.merge(
[DIAG] Candidate cities with >= 60 TEST windows: 311
  - Africa: 27
  - Asia: 35
  - 

In [8]:
# Save TIME outputs separately
!test -f runs_all.csv && mv runs_all.csv runs_all_time.csv
!test -f runs_by_evalset.csv && mv runs_by_evalset.csv runs_by_evalset_time.csv
!test -f boxplot_rmse_evalsets.png && mv boxplot_rmse_evalsets.png boxplot_rmse_evalsets_time.png
!test -f boxplot_energy_evalsets.png && mv boxplot_energy_evalsets.png boxplot_energy_evalsets_time.png

!rm -rf artifacts_time
!test -d artifacts && mv artifacts artifacts_time


In [9]:
# Final display: RMSE/MAE/etc for GEO vs TIME (mean ± std across the 5 sets)
import pandas as pd
import numpy as np

metrics = [
    "rmse_flat", "mae_flat", "crps_avg", "nll_per_h", "energy_joint", "energy_ind",
    "pit_ks", "cov95_avg", "iw95", "cal_ae", "corr_offdiag"
]

def summarize(csv_path: str, env_name: str):
    df = pd.read_csv(csv_path)
    out = []
    for m in metrics:
        if m not in df.columns:
            continue
        vals = df[m].to_numpy(dtype=float)
        out.append({
            "metric": m,
            f"{env_name}_mean": float(np.mean(vals)),
            f"{env_name}_std": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
        })
    return pd.DataFrame(out)

df_geo = summarize("runs_by_evalset_geo.csv", "geo")
df_time = summarize("runs_by_evalset_time.csv", "time")

df = df_geo.merge(df_time, on="metric", how="outer").sort_values("metric").reset_index(drop=True)

# display formatting
def fmt(mu, sd):
    if pd.isna(mu):
        return ""
    return f"{mu:.4f} ± {sd:.4f}"

df["GEO (mean±std)"] = [fmt(m,s) for m,s in zip(df["geo_mean"], df["geo_std"])]
df["TIME (mean±std)"] = [fmt(m,s) for m,s in zip(df["time_mean"], df["time_std"])]

display(df[["metric","GEO (mean±std)","TIME (mean±std)"]])

# save the table as well
df.to_csv("summary_env_metrics.csv", index=False)
print("Saved -> summary_env_metrics.csv")


,metric,GEO (mean±std),TIME (mean±std)
0,cal_ae,0.0497 ± 0.0089,0.0518 ± 0.0047
1,corr_offdiag,0.1795 ± 0.0356,0.1682 ± 0.0187
2,cov95_avg,94.2936 ± 1.5458,95.4419 ± 1.1588
3,crps_avg,0.8850 ± 0.0846,0.8856 ± 0.0833
4,energy_ind,2.6056 ± 0.2543,2.6009 ± 0.2520
5,energy_joint,2.6009 ± 0.2545,2.5956 ± 0.2523
6,iw95,6.2067 ± 0.5604,6.6371 ± 0.1809
7,mae_flat,1.2134 ± 0.1101,1.2127 ± 0.1110
8,nll_per_h,1.7684 ± 0.0763,1.7605 ± 0.0749
9,pit_ks,0.1298 ± 0.0297,0.1488 ± 0.0290


Saved -> summary_env_metrics.csv


In [10]:
# Bundle for download (artefacts + CSVs + summary)
!zip -r results_bundle.zip artifacts_geo artifacts_time runs_all_geo.csv runs_by_evalset_geo.csv runs_all_time.csv runs_by_evalset_time.csv summary_env_metrics.csv
from google.colab import files
files.download('results_bundle.zip')

  adding: artifacts_geo/ (stored 0%)
  adding: artifacts_geo/artifacts.zip (stored 0%)
  adding: artifacts_geo/tables/ (stored 0%)
  adding: artifacts_geo/tables/sign_mag_dependence_geo_set3_seedS1003_seedM44_H6.tex (deflated 68%)
  adding: artifacts_geo/tables/sign_mag_dependence_geo_set3_seedS1003_seedM43_H6.csv (deflated 60%)
  adding: artifacts_geo/tables/sign_mag_dependence_geo_set1_seedS1001_seedM43_H6.tex (deflated 69%)
  adding: artifacts_geo/tables/sign_mag_dependence_geo_set3_seedS1003_seedM42_H6.tex (deflated 69%)
  adding: artifacts_geo/tables/sign_mag_dependence_geo_set4_seedS1004_seedM44_H6.csv (deflated 60%)
  adding: artifacts_geo/tables/sign_mag_dependence_geo_set4_seedS1004_seedM43_H6.csv (deflated 60%)
  adding: artifacts_geo/tables/sign_mag_dependence_geo_set4_seedS1004_seedM42_H6.tex (deflated 68%)
  adding: artifacts_geo/tables/sign_mag_dependence_geo_set1_seedS1001_seedM42_H6.tex (deflated 69%)
  adding: artifacts_geo/tables/sign_mag_dependence_geo_set2_seedS1002

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
!python -m py_compile cnn_gru_spn_gamma_temperature_forecast.py

In [12]:
!grep -n "def make_windows_global_joint" -n cnn_gru_spn_gamma_temperature_forecast.py | head
!grep -n "env=\"geo\"" -n cnn_gru_spn_gamma_temperature_forecast.py | head

643:def make_windows_global_joint(
653:    env="geo",
